# SIACM ? treinamento reproduz?vel

Este notebook reproduz a metodologia de `train.py`: 80% para desenvolvimento, 20% para teste isolado, padroniza??o dentro do `Pipeline` e sele??o por F1 da classe maligna (`pos_label=0`). Estat?sticas e exemplos da interface s?o obtidos somente do desenvolvimento.


In [ ]:
from pathlib import Path
import json, joblib, numpy as np, pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, f1_score, accuracy_score, precision_score, recall_score, roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay
import matplotlib.pyplot as plt

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=[str(f) for f in data.feature_names])
y = pd.Series(data.target)
Xdev, Xtest, ydev, ytest = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)
scorer = make_scorer(f1_score, pos_label=0)

def pipe(model): return Pipeline([('scaler', StandardScaler()), ('model', model)])
grids = {
    'Regress?o Log?stica': (LogisticRegression(random_state=42, max_iter=1000), {'model__C':[.01,.1,1,10]}),
    '?rvore de Decis?o': (DecisionTreeClassifier(random_state=42), {'model__max_depth':[3,5,7,None], 'model__min_samples_split':[2,5,10]}),
    'Random Forest': (RandomForestClassifier(random_state=42), {'model__n_estimators':[50,100,200], 'model__max_depth':[5,10,None]}),
}
models, results = {}, {}
for name, (model, params) in grids.items():
    search = GridSearchCV(pipe(model), params, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring=scorer, n_jobs=-1)
    search.fit(Xdev, ydev); models[name] = search.best_estimator_
    pred = models[name].predict(Xtest); proba = models[name].predict_proba(Xtest)[:,0]
    results[name] = {'accuracy': accuracy_score(ytest,pred), 'precision_malignant': precision_score(ytest,pred,pos_label=0), 'recall_malignant': recall_score(ytest,pred,pos_label=0), 'f1_malignant': f1_score(ytest,pred,pos_label=0), 'auc': roc_auc_score(ytest==0,proba)}
pd.DataFrame(results).T.round(4)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, model in models.items():
    ConfusionMatrixDisplay.from_estimator(model, Xtest, ytest, ax=axes[0], values_format='d', colorbar=False)
    RocCurveDisplay.from_estimator(model, Xtest, ytest == 0, ax=axes[1], name=name)
axes[0].set_title('Matrizes de confus?o ? modelos atuais')
axes[1].set_title('Curvas ROC ? classe maligna')
plt.tight_layout()
plt.show()
